<a href="https://colab.research.google.com/github/jesila-mol18/AI-based-time-slot-Delivery-/blob/main/AI_Suggeted_Timeslot_using_RF.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.impute import SimpleImputer

# --------------------------
# 1. Load the Dataset
# --------------------------
# dataset_path = 'delivery_data.csv'
df = pd.read_csv('/content/drive/MyDrive/delivery_data.csv')

# Required columns in your dataset
required_columns = [
    'customer_id', 'day_of_week', 'preferred_time_slot', 'previous_successful_slot',
    'delivery_urgency', 'work_from_home', 'delivery_distance_km', 'delivery_frequency'
]
missing = [col for col in required_columns if col not in df.columns]
if missing:
    raise KeyError(f"Missing required columns: {', '.join(missing)}")

# --------------------------
# 2. Preprocess the Data
# --------------------------

# Map day_of_week to numeric (Monday=1, Tuesday=2, …, Sunday=7)
day_map = {'Monday': 1, 'Tuesday': 2, 'Wednesday': 3, 'Thursday': 4,
           'Friday': 5, 'Saturday': 6, 'Sunday': 7}
df['day_of_week_numeric'] = df['day_of_week'].map(day_map)

# Process previous_successful_slot: either map common labels or extract hour from "HH:MM"
time_slot_mapping = {'Morning': 9, 'Afternoon': 14, 'Evening': 18, 'Night': 21}
def process_previous_slot(slot):
    if pd.isnull(slot):
        return np.nan
    slot_str = str(slot)
    if slot_str in time_slot_mapping:
        return time_slot_mapping[slot_str]
    if ':' in slot_str:
        try:
            return int(slot_str.split(':')[0])
        except Exception:
            return np.nan
    return np.nan

df['previous_successful_slot_numeric'] = df['previous_successful_slot'].apply(process_previous_slot)

# Map delivery_urgency (Low=1, Medium=2, High=3)
urgency_map = {'Low': 1, 'Medium': 2, 'High': 3}
df['delivery_urgency_numeric'] = df['delivery_urgency'].map(urgency_map)

# Map work_from_home (Yes=1, No=0)
work_map = {'Yes': 1, 'No': 0}
df['work_from_home_numeric'] = df['work_from_home'].map(work_map)

# Ensure numeric conversion for delivery_distance_km and delivery_frequency
df['delivery_distance_km'] = pd.to_numeric(df['delivery_distance_km'], errors='coerce')
df['delivery_frequency'] = pd.to_numeric(df['delivery_frequency'], errors='coerce')

# Encode the target variable: preferred_time_slot
# (This converts labels like "Morning", "Night", etc. into numeric classes)
le = LabelEncoder()
df['preferred_time_slot_encoded'] = le.fit_transform(df['preferred_time_slot'].astype(str))

# Define the feature set for our AI model
feature_cols = [
    'day_of_week_numeric',
    'previous_successful_slot_numeric',
    'delivery_urgency_numeric',
    'work_from_home_numeric',
    'delivery_distance_km',
    'delivery_frequency'
]
X = df[feature_cols]
y = df['preferred_time_slot_encoded']

# Impute missing feature values using the median
imputer = SimpleImputer(strategy='median')
X_imputed = imputer.fit_transform(X)

# --------------------------
# 3. Train the AI Model
# --------------------------
model = RandomForestClassifier(random_state=42)
model.fit(X_imputed, y)

# --------------------------
# 4. AI-Driven Time Slot Suggestion
# --------------------------
# Use the median values of features as a representative sample for suggestion
median_features = np.median(X_imputed, axis=0).reshape(1, -1)
predicted_encoded = model.predict(median_features)[0]
suggested_slot = le.inverse_transform([predicted_encoded])[0]
print("AI Suggested Time Slot:", suggested_slot)



/usr/local/lib/python3.10/dist-packages/sklearn/impute/_base.py:635: UserWarning: Skipping features without any observed values: ['day_of_week_numeric' 'work_from_home_numeric']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(


AI Suggested Time Slot: Morning
